# LM2011 Phase 0 Validation Audit

Colab wrapper for `lm2011_phase0_validation_audit.py`. The audit code is sample-layout oriented, so this notebook prepares a small `/content` symlink layout that points to the Drive full-data folders discovered under `/content/drive/MyDrive/Data_LM`.

In [ ]:
from __future__ import annotations

import os
import subprocess
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
REPO_ENV_VAR = "NLP_THESIS_REPO_ROOT"
GIT_URL_ENV_VAR = "NLP_THESIS_GIT_URL"
GIT_REF_ENV_VAR = "NLP_THESIS_GIT_REF"
DEFAULT_GIT_URL = "https://github.com/Erew42/NLP_Thesis.git"
DEFAULT_GIT_REF = "main"
CLONE_ROOT = Path("/content/NLP_Thesis")


def _find_repo_root() -> Path | None:
    candidates: list[Path] = []
    env_root = os.environ.get(REPO_ENV_VAR)
    if env_root:
        candidates.append(Path(env_root).expanduser())

    cwd = Path.cwd().resolve()
    candidates.extend([cwd, *cwd.parents])

    if IN_COLAB:
        candidates.extend(
            [
                CLONE_ROOT,
                Path("/content/drive/MyDrive/NLP_Thesis"),
                Path("/content/drive/My Drive/NLP_Thesis"),
            ]
        )

    seen: set[Path] = set()
    for candidate in candidates:
        candidate = candidate.resolve()
        if candidate in seen:
            continue
        seen.add(candidate)
        if (candidate / "src" / "thesis_pkg" / "pipeline.py").exists():
            return candidate
    return None


if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive", force_remount=False)

repo_root = _find_repo_root()
if repo_root is None and IN_COLAB:
    git_url = os.environ.get(GIT_URL_ENV_VAR, DEFAULT_GIT_URL)
    if CLONE_ROOT.exists() and not (CLONE_ROOT / "src" / "thesis_pkg" / "pipeline.py").exists():
        raise FileExistsError(
            f"{CLONE_ROOT} exists but is not the NLP_Thesis repo. Remove it or set {REPO_ENV_VAR}."
        )
    if not CLONE_ROOT.exists():
        subprocess.check_call(["git", "clone", "--depth", "1", git_url, str(CLONE_ROOT)])
    repo_root = CLONE_ROOT

if repo_root is None:
    raise FileNotFoundError(
        "Could not locate the NLP_Thesis checkout. Run from the repo root, set NLP_THESIS_REPO_ROOT, or use Colab."
    )

repo_root = repo_root.resolve()
if IN_COLAB and (repo_root / ".git").exists() and os.environ.get("NLP_THESIS_SKIP_GIT_UPDATE", "0") != "1":
    git_ref = os.environ.get(GIT_REF_ENV_VAR, DEFAULT_GIT_REF)
    fetch_code = subprocess.call(["git", "-C", str(repo_root), "fetch", "--depth", "1", "origin", git_ref])
    checkout_target = "FETCH_HEAD" if fetch_code == 0 else git_ref
    subprocess.call(["git", "-C", str(repo_root), "checkout", checkout_target])

os.environ[REPO_ENV_VAR] = str(repo_root)
src = repo_root / "src"
if str(src) not in sys.path:
    sys.path.insert(0, str(src))

if IN_COLAB and os.environ.get("NLP_THESIS_SKIP_PIP_INSTALL", "0") != "1":
    install_target = f"{repo_root}[benchmark]"
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "--editable", install_target])

print({"IN_COLAB": IN_COLAB, "repo_root": str(repo_root), "src_exists": src.exists()})

In [ ]:
from pathlib import Path


def _resolve_colab_drive_root() -> Path:
    for candidate in (
        Path("/content/drive/MyDrive"),
        Path("/content/drive/My Drive"),
        Path("/content/drive"),
    ):
        if candidate.exists():
            return candidate
    return Path("/content/drive")


def _first_existing_path(*paths: Path) -> Path:
    for path in paths:
        if path.exists():
            return path
    return paths[0]


def _check_paths(paths: dict[str, Path], *, require_all: bool = True) -> None:
    rows = []
    missing = []
    for label, path in paths.items():
        exists = path.exists()
        rows.append({"label": label, "path": str(path), "exists": exists})
        if require_all and not exists:
            missing.append(f"{label}: {path}")
    try:
        import polars as pl

        display(pl.DataFrame(rows))
    except Exception:
        for row in rows:
            print(row)
    if missing:
        raise FileNotFoundError("Missing required Drive inputs:\n" + "\n".join(missing))

import datetime as dt
import os
import shutil

DRIVE_ROOT = _resolve_colab_drive_root()
WORK_ROOT = DRIVE_ROOT / "Data_LM"
SEC_CCM_RUN_ROOT = WORK_ROOT / "results" / "sec_ccm_unified_runner"

SOURCE_YEAR_MERGED_DIR = WORK_ROOT / "parquet_data" / "_year_merged"
DAILY_PANEL_PATH = WORK_ROOT / "CRSP_Compustat_data" / "derived_data" / "final_flagged_data_compdesc_added.parquet"
ADDITIONAL_DATA_DIR = _first_existing_path(
    WORK_ROOT / "LM2011_additional_data",
    repo_root / "full_data_run" / "LM2011_additional_data",
)
LM2011_OUTPUT_DIR = WORK_ROOT / "results" / "lm2011_sample_post_refinitiv_runner"
FINBERT_OUTPUT_ROOT = _first_existing_path(
    WORK_ROOT / "results" / "benchmarking" / "finbert_item_analysis_runner",
    WORK_ROOT / "results" / "benchmarking" / "finbert_sentence_parquet_inference",
)
OUTPUT_DIR = WORK_ROOT / "reports" / f"lm2011_phase0_validation_audit_{dt.date.today():%Y%m%d}"
LAYOUT_ROOT = Path("/content/lm2011_phase0_validation_layout") if IN_COLAB else repo_root / ".tmp" / "lm2011_phase0_validation_layout"

PACKETS = ["A", "B", "C", "D"]
YEAR_FILTER = None  # Example smoke audit: [2006, 2007, 2008]
FINBERT_RUN_DIR = None  # Set explicitly if the latest valid run should not be auto-selected.
MAX_EXAMPLE_ROWS = 25
SNIPPET_CHAR_LIMIT = 400
REGIME_COMPARE_DOC_LIMIT = 100

_check_paths(
    {
        "WORK_ROOT": WORK_ROOT,
        "SOURCE_YEAR_MERGED_DIR": SOURCE_YEAR_MERGED_DIR,
        "DAILY_PANEL_PATH": DAILY_PANEL_PATH,
        "ADDITIONAL_DATA_DIR": ADDITIONAL_DATA_DIR,
        "LM2011_OUTPUT_DIR": LM2011_OUTPUT_DIR,
        "FINBERT_OUTPUT_ROOT": FINBERT_OUTPUT_ROOT,
        "SEC_CCM_RUN_ROOT": SEC_CCM_RUN_ROOT,
    },
    require_all=False,
)

In [ ]:
def _reset_path(path: Path) -> None:
    if path.is_symlink() or path.is_file():
        path.unlink()
    elif path.exists():
        shutil.rmtree(path)


def _symlink_path(source: Path, target: Path, *, is_dir: bool) -> None:
    if not source.exists():
        raise FileNotFoundError(f"Cannot link missing source: {source}")
    target.parent.mkdir(parents=True, exist_ok=True)
    _reset_path(target)
    os.symlink(source, target, target_is_directory=is_dir)


LAYOUT_ROOT.mkdir(parents=True, exist_ok=True)
_symlink_path(SOURCE_YEAR_MERGED_DIR, LAYOUT_ROOT / "year_merged", is_dir=True)
_symlink_path(ADDITIONAL_DATA_DIR, LAYOUT_ROOT / "LM2011_additional_data", is_dir=True)
(LAYOUT_ROOT / "derived_data").mkdir(parents=True, exist_ok=True)
_symlink_path(
    DAILY_PANEL_PATH,
    LAYOUT_ROOT / "derived_data" / "final_flagged_data_compdesc_added.parquet",
    is_dir=False,
)

_check_paths(
    {
        "LAYOUT_ROOT": LAYOUT_ROOT,
        "LAYOUT_YEAR_MERGED": LAYOUT_ROOT / "year_merged",
        "LAYOUT_ADDITIONAL_DATA": LAYOUT_ROOT / "LM2011_additional_data",
        "LAYOUT_DAILY_PANEL": LAYOUT_ROOT / "derived_data" / "final_flagged_data_compdesc_added.parquet",
        "SEC_ITEMS_ANALYSIS": SEC_CCM_RUN_ROOT / "items_analysis",
    }
)

In [ ]:
from thesis_pkg.notebooks_and_scripts import lm2011_phase0_validation_audit as runner

RUN_ARGS = [
    "--sample-root", str(LAYOUT_ROOT),
    "--upstream-run-root", str(SEC_CCM_RUN_ROOT),
    "--lm2011-output-dir", str(LM2011_OUTPUT_DIR),
    "--finbert-output-root", str(FINBERT_OUTPUT_ROOT),
    "--output-dir", str(OUTPUT_DIR),
    "--packets", *PACKETS,
    "--max-example-rows", str(MAX_EXAMPLE_ROWS),
    "--snippet-char-limit", str(SNIPPET_CHAR_LIMIT),
    "--regime-compare-doc-limit", str(REGIME_COMPARE_DOC_LIMIT),
]
if YEAR_FILTER is not None:
    RUN_ARGS.extend(["--years", *(str(year) for year in YEAR_FILTER)])
if FINBERT_RUN_DIR is not None:
    RUN_ARGS.extend(["--finbert-run-dir", str(FINBERT_RUN_DIR)])

RUN_ARGS

In [ ]:
exit_code = runner.main(RUN_ARGS)
assert exit_code == 0

In [ ]:
import json

manifest_path = OUTPUT_DIR / "audit_manifest.json"
report_path = OUTPUT_DIR / "phase0_validation_report.md"
print({"manifest_path": str(manifest_path), "report_path": str(report_path)})

manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
print(json.dumps({"packet_statuses": manifest.get("packet_statuses"), "blocked_reasons": manifest.get("blocked_reasons")}, indent=2))

if report_path.exists():
    print(report_path.read_text(encoding="utf-8")[:4000])